In [ ]:
%pip install numpy pandas matplotlib scikit-learn lightgbm xgboost

In [ ]:
import pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
import lightgbm as lgb
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Ready.")

In [ ]:
import pickle
import numpy as np
from pathlib import Path

# ==============================
# Dataset path
# ==============================

DATA_DIR = Path(r"C:\\Users\\LENOVO\\Desktop\\FYP\\QuishingDataset")

QR_FILE = DATA_DIR / "qr_codes_29.pickle"
LABEL_FILE = DATA_DIR / "qr_codes_29_labels.pickle"

print("QR file:", QR_FILE)
print("Label file:", LABEL_FILE)

print("\nQR exists:", QR_FILE.exists())
print("Label exists:", LABEL_FILE.exists())


# ==============================
# Load dataset
# ==============================

with open(QR_FILE, "rb") as f:
    qr_codes = np.asarray(pickle.load(f))

with open(LABEL_FILE, "rb") as f:
    labels = np.asarray(pickle.load(f))


# ==============================
# Basic information
# ==============================

print("\nQR shape:", qr_codes.shape)
print("Label shape:", labels.shape)
print("QR dtype:", qr_codes.dtype)

unique_labels, counts = np.unique(labels, return_counts=True)

print(
    "Label counts:",
    dict(zip(unique_labels, counts))
)

In [ ]:
assert qr_codes.shape == (9987, 69, 69), qr_codes.shape
assert labels.shape == (9987,), labels.shape
assert set(np.unique(labels)).issubset({0,1})
print("Checks passed.")

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idxs = rng.choice(len(qr_codes), 10, replace=False)
fig, axes = plt.subplots(2,5,figsize=(12,5))
for ax, idx in zip(axes.ravel(), idxs):
    ax.imshow(qr_codes[idx], cmap="gray")
    ax.set_title(f"Label={labels[idx]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
X = qr_codes.reshape(qr_codes.shape[0], -1)
y = labels.astype(int)
print("X shape:", X.shape)
print("y shape:", y.shape)
assert X.shape == (9987, 4761)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)
print("Train:", X_train.shape)
print("Test :", X_test.shape)
print("Train labels:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test labels :", dict(zip(*np.unique(y_test, return_counts=True))))

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        C=0.1, solver="liblinear", max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3, min_samples_leaf=1,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        max_depth=20, n_estimators=100,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gaussian NB": GaussianNB(),
    "XGBoost": XGBClassifier(
        learning_rate=0.2,
        n_estimators=150,
        eval_metric="logloss",
        random_state=RANDOM_STATE
    ),
    "LightGBM": lgb.LGBMClassifier(
        learning_rate=0.1,
        n_estimators=200,
        random_state=RANDOM_STATE,
        verbosity=-1
    )
}

for name, m in models.items():
    print(name, "->", m)

In [ ]:
results=[]
predictions={}
probabilities={}
trained_models={}

# Scale only LR, following the public reproduction notebook.
scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train)
X_test_lr = scaler.transform(X_test)

for name, model in models.items():
    print("="*70)
    print("Training:", name)

    if name == "Logistic Regression":
        Xtr, Xte = X_train_lr, X_test_lr
    else:
        Xtr, Xte = X_train, X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:,1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_test, y_prob),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

    results.append(row)
    predictions[name]=y_pred
    probabilities[name]=y_prob
    trained_models[name]=model

    print({k: round(v,4) if isinstance(v,float) else v for k,v in row.items()})

results_df = pd.DataFrame(results)
display(results_df.round(4))

In [ ]:
for name in models:
    cm = confusion_matrix(y_test, predictions[name])
    fig, ax = plt.subplots(figsize=(4,4))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Benign (0)", "Phishing (1)"]
    ).plot(ax=ax, values_format="d")
    ax.set_title(name)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(8,6))
for name in models:
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    auc_v = roc_auc_score(y_test, probabilities[name])
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc_v:.4f})")

plt.plot([0,1],[0,1],linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Baseline Reproduction")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
published = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Gaussian NB",
        "XGBoost",
        "LightGBM"
    ],
    "Published_Accuracy":[0.7983,0.7578,0.7993,0.6376,0.8258,0.8293],
    "Published_Precision":[0.8129,0.7358,0.8570,0.8680,0.8332,0.8394],
    "Published_Recall":[0.7621,0.7856,0.7067,0.3036,0.8041,0.8041],
    "Published_F1":[0.7867,0.7599,0.7746,0.4498,0.8184,0.8214],
    "Published_AUC":[0.8737,0.8138,0.8908,0.6531,0.9083,0.9106]
})
display(published)

In [ ]:
rep = results_df[["Model","Accuracy","Precision","Recall","F1","AUC"]].copy()
comparison = published.merge(rep, on="Model", how="left")

comparison["ΔAccuracy"] = comparison["Accuracy"] - comparison["Published_Accuracy"]
comparison["ΔPrecision"] = comparison["Precision"] - comparison["Published_Precision"]
comparison["ΔRecall"] = comparison["Recall"] - comparison["Published_Recall"]
comparison["ΔF1"] = comparison["F1"] - comparison["Published_F1"]
comparison["ΔAUC"] = comparison["AUC"] - comparison["Published_AUC"]

display(comparison.round(4))

In [ ]:
report_table = comparison[[
    "Model",
    "Published_AUC","AUC","ΔAUC",
    "Published_F1","F1","ΔF1"
]].copy()

report_table.columns = [
    "Model",
    "Paper AUC","Reproduced AUC","AUC Difference",
    "Paper F1","Reproduced F1","F1 Difference"
]

display(report_table.round(4))

In [ ]:
OUT = Path("baseline_reproduction_results")
OUT.mkdir(exist_ok=True)

results_df.to_csv(OUT/"trad_chehab_reproduced_metrics.csv", index=False)
comparison.to_csv(OUT/"trad_chehab_paper_vs_reproduction.csv", index=False)

pred_df = pd.DataFrame({"y_true": y_test})
for name in models:
    safe = name.lower().replace(" ","_")
    pred_df[f"{safe}_pred"] = predictions[name]
    pred_df[f"{safe}_prob"] = probabilities[name]

pred_df.to_csv(OUT/"trad_chehab_test_predictions.csv", index=False)

print("Saved to:", OUT.resolve())
for p in OUT.iterdir():
    print("-", p.name)

## Interpretation note
If reproduced results are close but not identical to Table 2, this does not automatically mean the implementation is wrong.
The paper reports the 80/20 split, 10-fold CV for tuning, and final hyperparameters, but it does not report the exact random seed or exact train/test indices. Exact numerical replication is therefore not guaranteed from the paper alone.

In [ ]:
import gc
import random
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
DATA_DIR = Path(r"C:\\Users\\LENOVO\\Desktop\\FYP\\QuishingDataset")

QR_FILE = DATA_DIR / "qr_codes_29.pickle"
LABEL_FILE = DATA_DIR / "qr_codes_29_labels.pickle"

print("QR exists:", QR_FILE.exists(), QR_FILE)
print("Label exists:", LABEL_FILE.exists(), LABEL_FILE)

with open(QR_FILE, "rb") as f:
    qr_codes = np.asarray(pickle.load(f))

with open(LABEL_FILE, "rb") as f:
    labels = np.asarray(pickle.load(f)).astype(int)

print("QR shape:", qr_codes.shape)
print("Label shape:", labels.shape)
print("Class counts:", dict(zip(*np.unique(labels, return_counts=True))))

assert qr_codes.shape == (9987,69,69)
assert labels.shape == (9987,)

In [ ]:
X_raw = qr_codes.astype("float32")
X_filtered = np.empty_like(X_raw, dtype=np.float32)

for i in range(len(X_raw)):
    X_filtered[i] = gaussian_filter(X_raw[i], sigma=0.5)

mins = X_filtered.reshape(len(X_filtered), -1).min(axis=1).reshape(-1,1,1)
maxs = X_filtered.reshape(len(X_filtered), -1).max(axis=1).reshape(-1,1,1)

X_img = (X_filtered - mins) / np.maximum(maxs - mins, 1e-8)
X_img = X_img[..., np.newaxis].astype("float32")
y = labels.astype(int)

print("X:", X_img.shape)
print("range:", float(X_img.min()), float(X_img.max()))

In [ ]:
pixel_variance = X_img.reshape(len(X_img), -1).var(axis=1).astype("float32").reshape(-1,1)
print("metadata shape:", pixel_variance.shape)
display(pd.Series(pixel_variance.ravel()).describe())

In [ ]:
class GaussianNoiseClip(layers.Layer):
    def __init__(self, stddev=0.1, **kwargs):
        super().__init__(**kwargs)
        self.stddev = stddev

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.normal(tf.shape(inputs), mean=0.0, stddev=self.stddev, dtype=inputs.dtype)
            return tf.clip_by_value(inputs + noise, 0.0, 1.0)
        return inputs

def make_augmentation(seed):
    return keras.Sequential([
        layers.RandomRotation(
            factor=(-10/360, 10/360),
            fill_mode="nearest",
            seed=seed
        ),
        GaussianNoiseClip(stddev=0.1)
    ])

In [ ]:
L2_LAMBDA = 1e-5

def residual_block(x, filters=32, name="res"):
    shortcut = x

    y = layers.Conv2D(
        filters, 3, padding="same", use_bias=False,
        kernel_regularizer=regularizers.l2(L2_LAMBDA),
        name=f"{name}_conv1"
    )(x)
    y = layers.BatchNormalization(name=f"{name}_bn1")(y)
    y = layers.ReLU(name=f"{name}_relu1")(y)

    y = layers.Conv2D(
        filters, 3, padding="same", use_bias=False,
        kernel_regularizer=regularizers.l2(L2_LAMBDA),
        name=f"{name}_conv2"
    )(y)
    y = layers.BatchNormalization(name=f"{name}_bn2")(y)

    y = layers.Add(name=f"{name}_add")([shortcut, y])
    y = layers.ReLU(name=f"{name}_out")(y)
    return y

In [ ]:
def build_image_only_model(seed):
    aug = make_augmentation(seed)
    inp = keras.Input(shape=(69,69,1), name="image_input")
    x = aug(inp)

    x = layers.Conv2D(
        32,3,padding="same",use_bias=False,
        kernel_regularizer=regularizers.l2(L2_LAMBDA)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = residual_block(x,32,"block1")
    x = layers.MaxPooling2D(2)(x)

    x = residual_block(x,32,"block2")
    x = layers.MaxPooling2D(2)(x)

    x = residual_block(x,32,"block3")

    x = layers.Flatten()(x)
    x = layers.Dense(
        128, activation="relu",
        kernel_regularizer=regularizers.l2(L2_LAMBDA)
    )(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(1,activation="sigmoid")(x)
    return keras.Model(inp,out)

In [ ]:
def build_metadata_model(seed):
    aug = make_augmentation(seed)

    img_in = keras.Input(shape=(69,69,1), name="image_input")
    meta_in = keras.Input(shape=(1,), name="variance_input")

    x = aug(img_in)

    x = layers.Conv2D(
        32,3,padding="same",use_bias=False,
        kernel_regularizer=regularizers.l2(L2_LAMBDA)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = residual_block(x,32,"block1")
    x = layers.MaxPooling2D(2)(x)

    x = residual_block(x,32,"block2")
    x = layers.MaxPooling2D(2)(x)

    x = residual_block(x,32,"block3")

    x = layers.Flatten()(x)
    x = layers.Dense(
        128, activation="relu",
        kernel_regularizer=regularizers.l2(L2_LAMBDA)
    )(x)

    m = layers.Dense(8,activation="relu")(meta_in)

    x = layers.Concatenate()([x,m])
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(1,activation="sigmoid")(x)

    return keras.Model([img_in,meta_in],out)

In [ ]:
INITIAL_LR = 1e-3
MIN_LR = 1e-5
WEIGHT_DECAY = 1e-5
T0 = 10

BATCH_SIZE = 64
MAX_EPOCHS = 60
PATIENCE = 5

def compile_model(model, n_train):
    steps_per_epoch = int(np.ceil(n_train / BATCH_SIZE))
    first_decay_steps = steps_per_epoch * T0

    lr_schedule = keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=INITIAL_LR,
        first_decay_steps=first_decay_steps,
        t_mul=1.0,
        m_mul=1.0,
        alpha=MIN_LR/INITIAL_LR
    )

    optimizer = keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=WEIGHT_DECAY
    )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc")
        ]
    )
    return model

In [ ]:
N_SPLITS = 10

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

def evaluate_fold(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "Accuracy": accuracy_score(y_true,y_pred),
        "Precision": precision_score(y_true,y_pred,zero_division=0),
        "Recall": recall_score(y_true,y_pred,zero_division=0),
        "F1": f1_score(y_true,y_pred,zero_division=0),
        "AUC": roc_auc_score(y_true,y_prob),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

def run_cv(variant):
    rows=[]

    for fold,(train_idx,test_idx) in enumerate(skf.split(X_img,y),start=1):
        print("\n"+"="*70)
        print(f"{variant} | Fold {fold}/{N_SPLITS}")

        tf.keras.backend.clear_session()
        gc.collect()

        fold_seed = SEED + fold
        random.seed(fold_seed)
        np.random.seed(fold_seed)
        tf.random.set_seed(fold_seed)

        Xtr = X_img[train_idx]
        Xte = X_img[test_idx]
        ytr = y[train_idx]
        yte = y[test_idx]

        inner_train_idx, inner_val_idx = train_test_split(
            np.arange(len(Xtr)),
            test_size=0.10,
            random_state=fold_seed,
            stratify=ytr
        )

        if variant == "image_only":
            model = build_image_only_model(fold_seed)
            model = compile_model(model, len(inner_train_idx))

            train_input = Xtr[inner_train_idx]
            val_input = Xtr[inner_val_idx]
            test_input = Xte

        elif variant == "image_plus_metadata":
            model = build_metadata_model(fold_seed)
            model = compile_model(model, len(inner_train_idx))

            mtr = pixel_variance[train_idx]
            mte = pixel_variance[test_idx]

            train_input = [Xtr[inner_train_idx], mtr[inner_train_idx]]
            val_input = [Xtr[inner_val_idx], mtr[inner_val_idx]]
            test_input = [Xte, mte]

        else:
            raise ValueError("Unknown variant")

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor="val_auc",
                mode="max",
                patience=PATIENCE,
                restore_best_weights=True,
                verbose=0
            )
        ]

        hist = model.fit(
            train_input,
            ytr[inner_train_idx],
            validation_data=(val_input,ytr[inner_val_idx]),
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=callbacks,
            verbose=0
        )

        y_prob = model.predict(test_input,batch_size=BATCH_SIZE,verbose=0).ravel()
        metrics = evaluate_fold(yte,y_prob)
        metrics["Fold"] = fold
        metrics["Epochs"] = len(hist.history["loss"])
        rows.append(metrics)

        print(
            f"AUC={metrics['AUC']:.4f} | "
            f"F1={metrics['F1']:.4f} | "
            f"Accuracy={metrics['Accuracy']:.4f} | "
            f"Epochs={metrics['Epochs']}"
        )

    return pd.DataFrame(rows)

In [ ]:
image_only_results = run_cv("image_only")
display(image_only_results.round(4))

In [ ]:
metadata_results = run_cv("image_plus_metadata")
display(metadata_results.round(4))

In [ ]:
def summarize(df, name):
    rows=[]
    for metric in ["Accuracy","Precision","Recall","F1","AUC"]:
        rows.append({
            "Variant":name,
            "Metric":metric,
            "Mean":df[metric].mean(),
            "SD":df[metric].std(ddof=1),
            "Min":df[metric].min(),
            "Max":df[metric].max()
        })
    return pd.DataFrame(rows)

image_only_summary = summarize(image_only_results,"Image only")
metadata_summary = summarize(metadata_results,"Image + pixel variance")

display(image_only_summary.round(4))
display(metadata_summary.round(4))

In [ ]:
paper_acc = 0.930
paper_f1 = 0.920
paper_auc = 0.9313

comparison = pd.DataFrame([
    {
        "Variant":"Do et al. reported",
        "Accuracy Mean":paper_acc,
        "Accuracy SD":0.008,
        "F1 Mean":paper_f1,
        "F1 SD":np.nan,
        "AUC Mean":paper_auc,
        "AUC SD":np.nan
    },
    {
        "Variant":"Reproduction - image only",
        "Accuracy Mean":image_only_results["Accuracy"].mean(),
        "Accuracy SD":image_only_results["Accuracy"].std(ddof=1),
        "F1 Mean":image_only_results["F1"].mean(),
        "F1 SD":image_only_results["F1"].std(ddof=1),
        "AUC Mean":image_only_results["AUC"].mean(),
        "AUC SD":image_only_results["AUC"].std(ddof=1)
    },
    {
        "Variant":"Reproduction - image + pixel variance",
        "Accuracy Mean":metadata_results["Accuracy"].mean(),
        "Accuracy SD":metadata_results["Accuracy"].std(ddof=1),
        "F1 Mean":metadata_results["F1"].mean(),
        "F1 SD":metadata_results["F1"].std(ddof=1),
        "AUC Mean":metadata_results["AUC"].mean(),
        "AUC SD":metadata_results["AUC"].std(ddof=1)
    }
])

comparison["ΔAccuracy vs Paper"] = comparison["Accuracy Mean"] - paper_acc
comparison["ΔF1 vs Paper"] = comparison["F1 Mean"] - paper_f1
comparison["ΔAUC vs Paper"] = comparison["AUC Mean"] - paper_auc

display(comparison.round(4))

In [ ]:
gain = pd.DataFrame([{
    "Accuracy gain": metadata_results["Accuracy"].mean() - image_only_results["Accuracy"].mean(),
    "F1 gain": metadata_results["F1"].mean() - image_only_results["F1"].mean(),
    "AUC gain": metadata_results["AUC"].mean() - image_only_results["AUC"].mean()
}])

display(gain.round(4))

In [ ]:
OUT = Path("do_et_al_v2_reproduction_results")
OUT.mkdir(exist_ok=True)

image_only_results.to_csv(OUT/"image_only_10fold_results.csv",index=False)
metadata_results.to_csv(OUT/"image_plus_variance_10fold_results.csv",index=False)
image_only_summary.to_csv(OUT/"image_only_summary.csv",index=False)
metadata_summary.to_csv(OUT/"metadata_summary.csv",index=False)
comparison.to_csv(OUT/"paper_vs_reproduction.csv",index=False)
gain.to_csv(OUT/"metadata_gain.csv",index=False)

print("Saved to:", OUT.resolve())

We attempted a method-based reproduction of Do et al.'s Residual CNN using the reported dataset, preprocessing, augmentation, optimizer, regularization and 10-fold CV settings. However, the reproduced image-only model achieved AUC 0.8756 ± 0.0251, substantially below the reported 0.9313. Adding a reconstructed pixel-variance metadata branch further degraded performance. This indicates that key architectural and metadata-fusion details required for exact reproduction are not fully specified in the publication.

# Experiment — Matched Residual CNN Comparison

## Objective
Evaluate the previously implemented image-only Residual CNN
on the same 1,777-sample hostname-disjoint holdout used in
Experiment 6.

## Models
1. Trad-style raw-pixel XGBoost
2. Residual CNN
3. 10-feature lexical XGBoost

## Evaluation
ROC-AUC, F1, MCC, Accuracy, and paired comparison.

In [ ]:
%pip install numpy pandas matplotlib scikit-learn lightgbm xgboost

In [ ]:
import os

os.environ["LOKY_MAX_CPU_COUNT"] = "8"

In [ ]:
import pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
import lightgbm as lgb
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Ready.")

In [ ]:
import pickle
import numpy as np
from pathlib import Path

# ==============================
# Dataset path
# ==============================

DATA_DIR = Path(r"C:\\Users\\LENOVO\\Desktop\\FYP\\QuishingDataset")

QR_FILE = DATA_DIR / "qr_codes_29.pickle"
LABEL_FILE = DATA_DIR / "qr_codes_29_labels.pickle"

print("QR file:", QR_FILE)
print("Label file:", LABEL_FILE)

print("\nQR exists:", QR_FILE.exists())
print("Label exists:", LABEL_FILE.exists())


# ==============================
# Load dataset
# ==============================

with open(QR_FILE, "rb") as f:
    qr_codes = np.asarray(pickle.load(f))

with open(LABEL_FILE, "rb") as f:
    labels = np.asarray(pickle.load(f))


# ==============================
# Basic information
# ==============================

print("\nQR shape:", qr_codes.shape)
print("Label shape:", labels.shape)
print("QR dtype:", qr_codes.dtype)

unique_labels, counts = np.unique(labels, return_counts=True)

print(
    "Label counts:",
    dict(zip(unique_labels, counts))
)

In [ ]:
assert qr_codes.shape == (9987, 69, 69), qr_codes.shape
assert labels.shape == (9987,), labels.shape
assert set(np.unique(labels)).issubset({0,1})
print("Checks passed.")

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idxs = rng.choice(len(qr_codes), 10, replace=False)
fig, axes = plt.subplots(2,5,figsize=(12,5))
for ax, idx in zip(axes.ravel(), idxs):
    ax.imshow(qr_codes[idx], cmap="gray")
    ax.set_title(f"Label={labels[idx]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
X = qr_codes.reshape(qr_codes.shape[0], -1)
y = labels.astype(int)
print("X shape:", X.shape)
print("y shape:", y.shape)
assert X.shape == (9987, 4761)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)
print("Train:", X_train.shape)
print("Test :", X_test.shape)
print("Train labels:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test labels :", dict(zip(*np.unique(y_test, return_counts=True))))

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        C=0.1, solver="liblinear", max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3, min_samples_leaf=1,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        max_depth=20, n_estimators=100,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gaussian NB": GaussianNB(),
    "XGBoost": XGBClassifier(
        learning_rate=0.2,
        n_estimators=150,
        eval_metric="logloss",
        random_state=RANDOM_STATE
    ),
    "LightGBM": lgb.LGBMClassifier(
        learning_rate=0.1,
        n_estimators=200,
        random_state=RANDOM_STATE,
        verbosity=-1
    )
}

for name, m in models.items():
    print(name, "->", m)

In [ ]:
results=[]
predictions={}
probabilities={}
trained_models={}

# Scale only LR, following the public reproduction notebook.
scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train)
X_test_lr = scaler.transform(X_test)

for name, model in models.items():
    print("="*70)
    print("Training:", name)

    if name == "Logistic Regression":
        Xtr, Xte = X_train_lr, X_test_lr
    else:
        Xtr, Xte = X_train, X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:,1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_test, y_prob),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

    results.append(row)
    predictions[name]=y_pred
    probabilities[name]=y_prob
    trained_models[name]=model

    print({k: round(v,4) if isinstance(v,float) else v for k,v in row.items()})

results_df = pd.DataFrame(results)
display(results_df.round(4))

In [ ]:
for name in models:
    cm = confusion_matrix(y_test, predictions[name])
    fig, ax = plt.subplots(figsize=(4,4))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Benign (0)", "Phishing (1)"]
    ).plot(ax=ax, values_format="d")
    ax.set_title(name)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(8,6))
for name in models:
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    auc_v = roc_auc_score(y_test, probabilities[name])
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc_v:.4f})")

plt.plot([0,1],[0,1],linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Baseline Reproduction")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
published = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Gaussian NB",
        "XGBoost",
        "LightGBM"
    ],
    "Published_Accuracy":[0.7983,0.7578,0.7993,0.6376,0.8258,0.8293],
    "Published_Precision":[0.8129,0.7358,0.8570,0.8680,0.8332,0.8394],
    "Published_Recall":[0.7621,0.7856,0.7067,0.3036,0.8041,0.8041],
    "Published_F1":[0.7867,0.7599,0.7746,0.4498,0.8184,0.8214],
    "Published_AUC":[0.8737,0.8138,0.8908,0.6531,0.9083,0.9106]
})
display(published)

In [ ]:
rep = results_df[["Model","Accuracy","Precision","Recall","F1","AUC"]].copy()
comparison = published.merge(rep, on="Model", how="left")

comparison["ΔAccuracy"] = comparison["Accuracy"] - comparison["Published_Accuracy"]
comparison["ΔPrecision"] = comparison["Precision"] - comparison["Published_Precision"]
comparison["ΔRecall"] = comparison["Recall"] - comparison["Published_Recall"]
comparison["ΔF1"] = comparison["F1"] - comparison["Published_F1"]
comparison["ΔAUC"] = comparison["AUC"] - comparison["Published_AUC"]

display(comparison.round(4))

In [ ]:
report_table = comparison[[
    "Model",
    "Published_AUC","AUC","ΔAUC",
    "Published_F1","F1","ΔF1"
]].copy()

report_table.columns = [
    "Model",
    "Paper AUC","Reproduced AUC","AUC Difference",
    "Paper F1","Reproduced F1","F1 Difference"
]

display(report_table.round(4))

In [ ]:
OUT = Path("baseline_reproduction_results")
OUT.mkdir(exist_ok=True)

results_df.to_csv(OUT/"trad_chehab_reproduced_metrics.csv", index=False)
comparison.to_csv(OUT/"trad_chehab_paper_vs_reproduction.csv", index=False)

pred_df = pd.DataFrame({"y_true": y_test})
for name in models:
    safe = name.lower().replace(" ","_")
    pred_df[f"{safe}_pred"] = predictions[name]
    pred_df[f"{safe}_prob"] = probabilities[name]

pred_df.to_csv(OUT/"trad_chehab_test_predictions.csv", index=False)

print("Saved to:", OUT.resolve())
for p in OUT.iterdir():
    print("-", p.name)

In [ ]:
import gc
import random
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
DATA_DIR = Path(r"C:\\Users\\LENOVO\\Desktop\\FYP\\QuishingDataset")

QR_FILE = DATA_DIR / "qr_codes_29.pickle"
LABEL_FILE = DATA_DIR / "qr_codes_29_labels.pickle"

print("QR exists:", QR_FILE.exists(), QR_FILE)
print("Label exists:", LABEL_FILE.exists(), LABEL_FILE)

with open(QR_FILE, "rb") as f:
    qr_codes = np.asarray(pickle.load(f))

with open(LABEL_FILE, "rb") as f:
    labels = np.asarray(pickle.load(f)).astype(int)

print("QR shape:", qr_codes.shape)
print("Label shape:", labels.shape)
print("Class counts:", dict(zip(*np.unique(labels, return_counts=True))))

assert qr_codes.shape == (9987,69,69)
assert labels.shape == (9987,)

In [ ]:
X_raw = qr_codes.astype("float32")
X_filtered = np.empty_like(X_raw, dtype=np.float32)

for i in range(len(X_raw)):
    X_filtered[i] = gaussian_filter(X_raw[i], sigma=0.5)

mins = X_filtered.reshape(len(X_filtered), -1).min(axis=1).reshape(-1,1,1)
maxs = X_filtered.reshape(len(X_filtered), -1).max(axis=1).reshape(-1,1,1)

X_img = (X_filtered - mins) / np.maximum(maxs - mins, 1e-8)
X_img = X_img[..., np.newaxis].astype("float32")
y = labels.astype(int)

print("X:", X_img.shape)
print("range:", float(X_img.min()), float(X_img.max()))

In [ ]:
pixel_variance = X_img.reshape(len(X_img), -1).var(axis=1).astype("float32").reshape(-1,1)
print("metadata shape:", pixel_variance.shape)
display(pd.Series(pixel_variance.ravel()).describe())

In [ ]:
class GaussianNoiseClip(layers.Layer):
    def __init__(self, stddev=0.1, **kwargs):
        super().__init__(**kwargs)
        self.stddev = stddev

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.normal(tf.shape(inputs), mean=0.0, stddev=self.stddev, dtype=inputs.dtype)
            return tf.clip_by_value(inputs + noise, 0.0, 1.0)
        return inputs

def make_augmentation(seed):
    return keras.Sequential([
        layers.RandomRotation(
            factor=(-10/360, 10/360),
            fill_mode="nearest",
            seed=seed
        ),
        GaussianNoiseClip(stddev=0.1)
    ])

In [ ]:
L2_LAMBDA = 1e-5

def residual_block(x, filters=32, name="res"):
    shortcut = x

    y = layers.Conv2D(
        filters, 3, padding="same", use_bias=False,
        kernel_regularizer=regularizers.l2(L2_LAMBDA),
        name=f"{name}_conv1"
    )(x)
    y = layers.BatchNormalization(name=f"{name}_bn1")(y)
    y = layers.ReLU(name=f"{name}_relu1")(y)

    y = layers.Conv2D(
        filters, 3, padding="same", use_bias=False,
        kernel_regularizer=regularizers.l2(L2_LAMBDA),
        name=f"{name}_conv2"
    )(y)
    y = layers.BatchNormalization(name=f"{name}_bn2")(y)

    y = layers.Add(name=f"{name}_add")([shortcut, y])
    y = layers.ReLU(name=f"{name}_out")(y)
    return y

In [ ]:
def build_image_only_model(seed):
    aug = make_augmentation(seed)
    inp = keras.Input(shape=(69,69,1), name="image_input")
    x = aug(inp)

    x = layers.Conv2D(
        32,3,padding="same",use_bias=False,
        kernel_regularizer=regularizers.l2(L2_LAMBDA)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = residual_block(x,32,"block1")
    x = layers.MaxPooling2D(2)(x)

    x = residual_block(x,32,"block2")
    x = layers.MaxPooling2D(2)(x)

    x = residual_block(x,32,"block3")

    x = layers.Flatten()(x)
    x = layers.Dense(
        128, activation="relu",
        kernel_regularizer=regularizers.l2(L2_LAMBDA)
    )(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(1,activation="sigmoid")(x)
    return keras.Model(inp,out)

In [ ]:
def build_metadata_model(seed):
    aug = make_augmentation(seed)

    img_in = keras.Input(shape=(69,69,1), name="image_input")
    meta_in = keras.Input(shape=(1,), name="variance_input")

    x = aug(img_in)

    x = layers.Conv2D(
        32,3,padding="same",use_bias=False,
        kernel_regularizer=regularizers.l2(L2_LAMBDA)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = residual_block(x,32,"block1")
    x = layers.MaxPooling2D(2)(x)

    x = residual_block(x,32,"block2")
    x = layers.MaxPooling2D(2)(x)

    x = residual_block(x,32,"block3")

    x = layers.Flatten()(x)
    x = layers.Dense(
        128, activation="relu",
        kernel_regularizer=regularizers.l2(L2_LAMBDA)
    )(x)

    m = layers.Dense(8,activation="relu")(meta_in)

    x = layers.Concatenate()([x,m])
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(1,activation="sigmoid")(x)

    return keras.Model([img_in,meta_in],out)

In [ ]:
INITIAL_LR = 1e-3
MIN_LR = 1e-5
WEIGHT_DECAY = 1e-5
T0 = 10

BATCH_SIZE = 64
MAX_EPOCHS = 60
PATIENCE = 5

def compile_model(model, n_train):
    steps_per_epoch = int(np.ceil(n_train / BATCH_SIZE))
    first_decay_steps = steps_per_epoch * T0

    lr_schedule = keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=INITIAL_LR,
        first_decay_steps=first_decay_steps,
        t_mul=1.0,
        m_mul=1.0,
        alpha=MIN_LR/INITIAL_LR
    )

    optimizer = keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=WEIGHT_DECAY
    )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc")
        ]
    )
    return model

In [ ]:
N_SPLITS = 10

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

def evaluate_fold(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "Accuracy": accuracy_score(y_true,y_pred),
        "Precision": precision_score(y_true,y_pred,zero_division=0),
        "Recall": recall_score(y_true,y_pred,zero_division=0),
        "F1": f1_score(y_true,y_pred,zero_division=0),
        "AUC": roc_auc_score(y_true,y_prob),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

def run_cv(variant):
    rows=[]

    for fold,(train_idx,test_idx) in enumerate(skf.split(X_img,y),start=1):
        print("\n"+"="*70)
        print(f"{variant} | Fold {fold}/{N_SPLITS}")

        tf.keras.backend.clear_session()
        gc.collect()

        fold_seed = SEED + fold
        random.seed(fold_seed)
        np.random.seed(fold_seed)
        tf.random.set_seed(fold_seed)

        Xtr = X_img[train_idx]
        Xte = X_img[test_idx]
        ytr = y[train_idx]
        yte = y[test_idx]

        inner_train_idx, inner_val_idx = train_test_split(
            np.arange(len(Xtr)),
            test_size=0.10,
            random_state=fold_seed,
            stratify=ytr
        )

        if variant == "image_only":
            model = build_image_only_model(fold_seed)
            model = compile_model(model, len(inner_train_idx))

            train_input = Xtr[inner_train_idx]
            val_input = Xtr[inner_val_idx]
            test_input = Xte

        elif variant == "image_plus_metadata":
            model = build_metadata_model(fold_seed)
            model = compile_model(model, len(inner_train_idx))

            mtr = pixel_variance[train_idx]
            mte = pixel_variance[test_idx]

            train_input = [Xtr[inner_train_idx], mtr[inner_train_idx]]
            val_input = [Xtr[inner_val_idx], mtr[inner_val_idx]]
            test_input = [Xte, mte]

        else:
            raise ValueError("Unknown variant")

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor="val_auc",
                mode="max",
                patience=PATIENCE,
                restore_best_weights=True,
                verbose=0
            )
        ]

        hist = model.fit(
            train_input,
            ytr[inner_train_idx],
            validation_data=(val_input,ytr[inner_val_idx]),
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=callbacks,
            verbose=0
        )

        y_prob = model.predict(test_input,batch_size=BATCH_SIZE,verbose=0).ravel()
        metrics = evaluate_fold(yte,y_prob)
        metrics["Fold"] = fold
        metrics["Epochs"] = len(hist.history["loss"])
        rows.append(metrics)

        print(
            f"AUC={metrics['AUC']:.4f} | "
            f"F1={metrics['F1']:.4f} | "
            f"Accuracy={metrics['Accuracy']:.4f} | "
            f"Epochs={metrics['Epochs']}"
        )

    return pd.DataFrame(rows)

In [ ]:
image_only_results = run_cv("image_only")
display(image_only_results.round(4))

In [ ]:
metadata_results = run_cv("image_plus_metadata")
display(metadata_results.round(4))

In [ ]:
def summarize(df, name):
    rows=[]
    for metric in ["Accuracy","Precision","Recall","F1","AUC"]:
        rows.append({
            "Variant":name,
            "Metric":metric,
            "Mean":df[metric].mean(),
            "SD":df[metric].std(ddof=1),
            "Min":df[metric].min(),
            "Max":df[metric].max()
        })
    return pd.DataFrame(rows)

image_only_summary = summarize(image_only_results,"Image only")
metadata_summary = summarize(metadata_results,"Image + pixel variance")

display(image_only_summary.round(4))
display(metadata_summary.round(4))

In [ ]:
paper_acc = 0.930
paper_f1 = 0.920
paper_auc = 0.9313

comparison = pd.DataFrame([
    {
        "Variant":"Do et al. reported",
        "Accuracy Mean":paper_acc,
        "Accuracy SD":0.008,
        "F1 Mean":paper_f1,
        "F1 SD":np.nan,
        "AUC Mean":paper_auc,
        "AUC SD":np.nan
    },
    {
        "Variant":"Reproduction - image only",
        "Accuracy Mean":image_only_results["Accuracy"].mean(),
        "Accuracy SD":image_only_results["Accuracy"].std(ddof=1),
        "F1 Mean":image_only_results["F1"].mean(),
        "F1 SD":image_only_results["F1"].std(ddof=1),
        "AUC Mean":image_only_results["AUC"].mean(),
        "AUC SD":image_only_results["AUC"].std(ddof=1)
    },
    {
        "Variant":"Reproduction - image + pixel variance",
        "Accuracy Mean":metadata_results["Accuracy"].mean(),
        "Accuracy SD":metadata_results["Accuracy"].std(ddof=1),
        "F1 Mean":metadata_results["F1"].mean(),
        "F1 SD":metadata_results["F1"].std(ddof=1),
        "AUC Mean":metadata_results["AUC"].mean(),
        "AUC SD":metadata_results["AUC"].std(ddof=1)
    }
])

comparison["ΔAccuracy vs Paper"] = comparison["Accuracy Mean"] - paper_acc
comparison["ΔF1 vs Paper"] = comparison["F1 Mean"] - paper_f1
comparison["ΔAUC vs Paper"] = comparison["AUC Mean"] - paper_auc

display(comparison.round(4))

In [ ]:
gain = pd.DataFrame([{
    "Accuracy gain": metadata_results["Accuracy"].mean() - image_only_results["Accuracy"].mean(),
    "F1 gain": metadata_results["F1"].mean() - image_only_results["F1"].mean(),
    "AUC gain": metadata_results["AUC"].mean() - image_only_results["AUC"].mean()
}])

display(gain.round(4))

In [ ]:
OUT = Path("do_et_al_v2_reproduction_results")
OUT.mkdir(exist_ok=True)

image_only_results.to_csv(OUT/"image_only_10fold_results.csv",index=False)
metadata_results.to_csv(OUT/"image_plus_variance_10fold_results.csv",index=False)
image_only_summary.to_csv(OUT/"image_only_summary.csv",index=False)
metadata_summary.to_csv(OUT/"metadata_summary.csv",index=False)
comparison.to_csv(OUT/"paper_vs_reproduction.csv",index=False)
gain.to_csv(OUT/"metadata_gain.csv",index=False)

print("Saved to:", OUT.resolve())

# PART 3 — Matched Residual CNN Comparison (Independent, self-contained)

In [ ]:
from pathlib import Path
import os, gc, json, pickle, random, time, platform

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                             precision_score, recall_score, matthews_corrcoef,
                             confusion_matrix)

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, regularizers
except ImportError as exc:
    raise ImportError('TensorFlow is missing. In Jupyter run `%pip install tensorflow` then restart the kernel.') from exc

P3_SEEDS = (42, 43, 44)
P3_SPLIT_SEED = 42
P3_VAL_SPLIT_SEED = 20261108
P3_VAL_CANDIDATES = 200
P3_VAL_FRACTION = 0.12
P3_BATCH_SIZE = 64
P3_MAX_EPOCHS = 60
P3_PATIENCE = 12
P3_INITIAL_LR = 3e-4
P3_BOOTSTRAPS = 300
P3_THRESHOLD = 0.5
P3_OUT = Path.cwd() / 'part3_cnn_matched_clean_results'

# Optional exact locations. Leave as None when the files are discoverable.
P3_PATHS = {
    'qr_codes_29.pickle': None,
    'qr_codes_29_labels.pickle': None,
    'qr_full_feature_dataset_v2.csv': None,
    'matched_holdout_predictions.csv': None,
}

P3_HOME = Path.home()
P3_ROOTS = [
    Path.cwd(), Path.cwd().parent,
    Path.cwd() / 'experiment6_matched_trad_vs_lexical',
    Path.cwd().parent / 'experiment6_matched_trad_vs_lexical',
    P3_HOME / 'Desktop' / 'FYP',
    P3_HOME / 'Desktop' / 'FYP' / 'QuishingDataset',
    P3_HOME / 'OneDrive' / 'Desktop' / 'FYP',
    P3_HOME / 'OneDrive' / 'Desktop' / 'FYP' / 'QuishingDataset',
    P3_HOME / 'Documents', P3_HOME / 'Downloads',
    Path(r'C:\Users\LENOVO\Desktop\FYP'),
    Path(r'C:\Users\LENOVO\Desktop\FYP\QuishingDataset'),
    Path(r'C:\Users\LENOVO\FYP\QuishingDataset'),
]

def p3_find(filename):
    explicit = P3_PATHS[filename]
    if explicit:
        p = Path(explicit).expanduser()
        if not p.is_file():
            raise FileNotFoundError(f'Incorrect P3_PATHS[{filename!r}]: {p}')
        return p.resolve()
    for root in P3_ROOTS:
        for p in (root / filename,
                  root / 'experiment6_matched_trad_vs_lexical' / filename,
                  root / 'QuishingDataset' / filename):
            if p.is_file():
                return p.resolve()
    raise FileNotFoundError(
        f'Cannot locate {filename}. Edit P3_PATHS[{filename!r}] in Section 3.1. '
        f'Jupyter working directory: {Path.cwd()}')

P3_FILES = {name: p3_find(name) for name in P3_PATHS}
for name, path in P3_FILES.items():
    print(f'INPUT {name}: {path}')
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
print('NEW output directory:', P3_OUT)

In [ ]:
with open(P3_FILES['qr_codes_29.pickle'], 'rb') as fh:
    p3_qr = np.asarray(pickle.load(fh))
with open(P3_FILES['qr_codes_29_labels.pickle'], 'rb') as fh:
    p3_qr_y = np.asarray(pickle.load(fh), dtype=int)
p3_df = pd.read_csv(P3_FILES['qr_full_feature_dataset_v2.csv'])
p3_exp6 = pd.read_csv(P3_FILES['matched_holdout_predictions.csv'])

assert p3_qr.shape == (9987, 69, 69), f'Unexpected QR shape {p3_qr.shape}'
assert p3_qr_y.shape == (9987,), f'Unexpected original label shape: {p3_qr_y.shape}'
assert len(p3_df) == 9713, f'Unexpected aligned data length: {len(p3_df)}'
for col in ('original_index', 'hostname', 'label'):
    assert col in p3_df, f'Missing aligned data column: {col}'
assert p3_df['original_index'].is_unique
assert p3_df[['original_index', 'hostname', 'label']].notna().all().all()

p3_ids = p3_df['original_index'].to_numpy(dtype=int)
p3_y = p3_df['label'].to_numpy(dtype=int)
p3_hosts = p3_df['hostname'].astype(str).to_numpy()
assert p3_ids.min() >= 0 and p3_ids.max() < len(p3_qr)
assert np.array_equal(p3_qr_y[p3_ids], p3_y), 'QR-to-URL label mismatch.'
assert np.array_equal(np.unique(p3_y), np.array([0, 1]))

p3_outer = GroupShuffleSplit(n_splits=1, test_size=0.20,
                             random_state=P3_SPLIT_SEED)
p3_dev_ix, p3_test_ix = next(p3_outer.split(p3_df, p3_y, groups=p3_hosts))
assert (len(p3_dev_ix), len(p3_test_ix)) == (7936, 1777), (
    len(p3_dev_ix), len(p3_test_ix))
assert set(p3_hosts[p3_dev_ix]).isdisjoint(set(p3_hosts[p3_test_ix]))

# Exp6 predictions are the authoritative row-level reference.
p3_test_ids = p3_ids[p3_test_ix]
assert len(p3_exp6) == 1777 and p3_exp6['original_index'].is_unique
assert set(p3_test_ids) == set(p3_exp6['original_index'].to_numpy(dtype=int)), (
    'The Experiment 6 holdout indices differ. STOP. Do not compare mismatched sets.')
p3_exp6 = (p3_exp6.set_index('original_index')
           .loc[p3_test_ids].reset_index())
assert np.array_equal(p3_exp6['label'].to_numpy(dtype=int), p3_y[p3_test_ix])
assert np.array_equal(p3_exp6['hostname'].astype(str).to_numpy(),
                      p3_hosts[p3_test_ix])

p3_y_dev = p3_y[p3_dev_ix]
p3_hosts_dev = p3_hosts[p3_dev_ix]
p3_y_test = p3_y[p3_test_ix]
p3_hosts_test = p3_hosts[p3_test_ix]
p3_raw_probs = p3_exp6['raw_pixel_probability'].to_numpy(dtype=float)
p3_lex_probs = p3_exp6['lexical_probability'].to_numpy(dtype=float)
p3_raw_auc = roc_auc_score(p3_y_test, p3_raw_probs)
p3_lex_auc = roc_auc_score(p3_y_test, p3_lex_probs)
assert abs(p3_raw_auc - 0.88218) < 0.0002, f'Unexpected raw AUC: {p3_raw_auc}'
assert abs(p3_lex_auc - 0.95095) < 0.0002, f'Unexpected lexical AUC: {p3_lex_auc}'
print(f'PASS: Exact Exp6 split: dev={len(p3_dev_ix)}, test={len(p3_test_ix)}')
print('PASS: All 1,777 row IDs, hostnames, labels, reference probabilities match.')
print(f'PASS: raw AUC={p3_raw_auc:.5f}, lexical AUC={p3_lex_auc:.5f}')

In [ ]:
p3_candidate_splitter = GroupShuffleSplit(
    n_splits=P3_VAL_CANDIDATES, test_size=P3_VAL_FRACTION,
    random_state=P3_VAL_SPLIT_SEED)
p3_candidates = []
for tr, va in p3_candidate_splitter.split(
        np.zeros(len(p3_y_dev)), p3_y_dev, groups=p3_hosts_dev):
    tr_rate = float(p3_y_dev[tr].mean())
    va_rate = float(p3_y_dev[va].mean())
    va_fraction = len(va) / len(p3_y_dev)
    # Match class prevalences first, then target validation fraction.
    score = abs(tr_rate - va_rate) + 0.25 * abs(va_fraction - P3_VAL_FRACTION)
    p3_candidates.append((score, tr, va))
_, p3_inner_tr, p3_inner_va = min(p3_candidates, key=lambda v: v[0])
assert set(p3_hosts_dev[p3_inner_tr]).isdisjoint(
    set(p3_hosts_dev[p3_inner_va]))
assert set(p3_hosts_dev).isdisjoint(set(p3_hosts_test))
assert len(np.unique(p3_y_dev[p3_inner_tr])) == 2
assert len(np.unique(p3_y_dev[p3_inner_va])) == 2

print('=== FIXED INNER SPLIT (same for seeds 42, 43, 44) ===')
print('Train n:',len(p3_inner_tr),'| validation n:',len(p3_inner_va))
print('Train phishing rate:',round(float(p3_y_dev[p3_inner_tr].mean()),4))
print('Valid phishing rate:',round(float(p3_y_dev[p3_inner_va].mean()),4))
print('Hostname overlap:',len(set(p3_hosts_dev[p3_inner_tr]) &
                              set(p3_hosts_dev[p3_inner_va])))

P3_OUT.mkdir(parents=True, exist_ok=True)
pd.DataFrame({
    'original_index':p3_ids[p3_dev_ix[p3_inner_va]],
    'hostname':p3_hosts_dev[p3_inner_va],
    'label':p3_y_dev[p3_inner_va],
}).to_csv(P3_OUT/'fixed_inner_validation_ids.csv', index=False)
pd.DataFrame({
    'original_index':p3_test_ids,
    'hostname':p3_hosts_test,
    'label':p3_y_test,
}).to_csv(P3_OUT/'matched_holdout_ids.csv', index=False)

In [ ]:
def p3_preprocess_qr(batch):
    batch = np.asarray(batch, dtype=np.float32)
    filtered = gaussian_filter(batch, sigma=(0, 0.5, 0.5))
    min_val = filtered.min(axis=(1, 2), keepdims=True)
    max_val = filtered.max(axis=(1, 2), keepdims=True)
    return ((filtered-min_val) / np.maximum(max_val-min_val, 1e-8))[...,None].astype('float32')

p3_X_dev = p3_preprocess_qr(p3_qr[p3_ids[p3_dev_ix]])
p3_X_test = p3_preprocess_qr(p3_qr[p3_test_ids])
assert p3_X_dev.shape == (7936,69,69,1)
assert p3_X_test.shape == (1777,69,69,1)
assert np.isfinite(p3_X_dev).all() and np.isfinite(p3_X_test).all()
assert 0 <= p3_X_dev.min() <= p3_X_dev.max() <= 1.0
print('CNN shapes:',p3_X_dev.shape,p3_X_test.shape)

class P3_GaussianNoiseClip(layers.Layer):
    def __init__(self, stddev=0.1, **kwargs):
        super().__init__(**kwargs)
        self.stddev=stddev
    def call(self, inputs, training=None):
        if training:
            noise=tf.random.normal(tf.shape(inputs),mean=0.,stddev=self.stddev,
                                   dtype=inputs.dtype)
            return tf.clip_by_value(inputs+noise,0.,1.)
        return inputs

def p3_augmentation(seed):
    return keras.Sequential([
        layers.RandomRotation(factor=(-10/360,10/360),
                              fill_mode='nearest',seed=seed),
        P3_GaussianNoiseClip(stddev=0.1),
    ])

def p3_residual_block(x,filters=32,name='res'):
    shortcut=x
    y=layers.Conv2D(filters,3,padding='same',use_bias=False,
        kernel_regularizer=regularizers.l2(1e-5),name=f'{name}_conv1')(x)
    y=layers.BatchNormalization(name=f'{name}_bn1')(y)
    y=layers.ReLU(name=f'{name}_relu1')(y)
    y=layers.Conv2D(filters,3,padding='same',use_bias=False,
        kernel_regularizer=regularizers.l2(1e-5),name=f'{name}_conv2')(y)
    y=layers.BatchNormalization(name=f'{name}_bn2')(y)
    y=layers.Add(name=f'{name}_add')([shortcut,y])
    return layers.ReLU(name=f'{name}_out')(y)

def p3_build_cnn(seed):
    inp=keras.Input(shape=(69,69,1),name='image_input')
    x=p3_augmentation(seed)(inp)
    x=layers.Conv2D(32,3,padding='same',use_bias=False,
        kernel_regularizer=regularizers.l2(1e-5))(x)
    x=layers.BatchNormalization()(x)
    x=layers.ReLU()(x)
    x=p3_residual_block(x,32,'block1')
    x=layers.MaxPooling2D(2)(x)
    x=p3_residual_block(x,32,'block2')
    x=layers.MaxPooling2D(2)(x)
    x=p3_residual_block(x,32,'block3')
    x=layers.Flatten()(x)
    x=layers.Dense(128,activation='relu',
                   kernel_regularizer=regularizers.l2(1e-5))(x)
    x=layers.Dropout(0.3)(x)
    out=layers.Dense(1,activation='sigmoid')(x)
    return keras.Model(inp,out,name='image_only_residual_cnn')

def p3_compile_cnn(model):
    # Stable-training diagnostic protocol; deliberately NOT the old cosine refit.
    optimizer=keras.optimizers.AdamW(learning_rate=P3_INITIAL_LR,
                                    weight_decay=1e-5,clipnorm=1.0)
    model.compile(optimizer=optimizer,loss='binary_crossentropy',
        metrics=[keras.metrics.BinaryAccuracy(name='accuracy'),
                 keras.metrics.AUC(name='auc')])
    return model

In [ ]:
def p3_metrics(y,prob,threshold=P3_THRESHOLD):
    prob=np.asarray(prob,dtype=float).reshape(-1)
    pred=(prob >= threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(auc=float(roc_auc_score(y,prob)),
        f1=float(f1_score(y,pred,zero_division=0)),
        mcc=float(matthews_corrcoef(y,pred)),
        accuracy=float(accuracy_score(y,pred)),
        precision=float(precision_score(y,pred,zero_division=0)),
        recall=float(recall_score(y,pred,zero_division=0)),
        predicted_phishing_rate=float(pred.mean()),
        probability_std=float(prob.std()),
        tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))

p3_seed_rows=[]
p3_preds=[]
p3_histories={}
p3_scores={}

for seed in P3_SEEDS:
    print('\n'+'='*68)
    print(f'CNN: seed {seed}, fixed inner split, no refit')
    keras.backend.clear_session()
    gc.collect()
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    model=p3_compile_cnn(p3_build_cnn(seed))
    callbacks=[
        keras.callbacks.TerminateOnNaN(),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc',mode='max',factor=0.5,patience=4,
            min_lr=1e-5,verbose=0),
        keras.callbacks.EarlyStopping(
            monitor='val_auc',mode='max',patience=P3_PATIENCE,
            restore_best_weights=True,verbose=0),
    ]
    start=time.perf_counter()
    history=model.fit(
        p3_X_dev[p3_inner_tr],p3_y_dev[p3_inner_tr],
        validation_data=(p3_X_dev[p3_inner_va],p3_y_dev[p3_inner_va]),
        batch_size=P3_BATCH_SIZE,epochs=P3_MAX_EPOCHS,
        callbacks=callbacks,verbose=2,shuffle=True)
    elapsed=time.perf_counter()-start
    h=pd.DataFrame(history.history)
    h.to_csv(P3_OUT/f'cnn_history_seed_{seed}.csv',index=False)
    p3_histories[seed]=h
    best_epoch=int(h['val_auc'].idxmax())+1

    # Check the selected model on INTERNAL validation BEFORE touching holdout.
    vprob=model.predict(p3_X_dev[p3_inner_va],
                        batch_size=P3_BATCH_SIZE,verbose=0).reshape(-1)
    tprob=model.predict(p3_X_dev[p3_inner_tr],
                        batch_size=P3_BATCH_SIZE,verbose=0).reshape(-1)
    val_auc=float(roc_auc_score(p3_y_dev[p3_inner_va],vprob))
    train_auc=float(roc_auc_score(p3_y_dev[p3_inner_tr],tprob))

    # Once per seed, run the exactly aligned matched holdout.
    proba=np.asarray(model.predict(p3_X_test,
                batch_size=P3_BATCH_SIZE,verbose=0).reshape(-1),dtype=float)
    assert len(proba)==1777 and np.isfinite(proba).all()
    assert ((proba>=0)&(proba<=1)).all()
    met=p3_metrics(p3_y_test,proba)
    print(f'Seed {seed}: train AUC {train_auc:.4f}, '
          f'val AUC {val_auc:.4f}, test AUC {met["auc"]:.4f}, '
          f'F1 {met["f1"]:.4f}, MCC {met["mcc"]:.4f}, '
          f'prob std {met["probability_std"]:.6f}')
    if met['probability_std'] < 0.01 or val_auc < 0.60:
        print('WARNING: This seed may have collapsed; KEEP it in the report.')
    p3_scores[seed]=proba.copy()
    p3_seed_rows.append(dict(seed=seed,n=1777,
        train_n=len(p3_inner_tr),validation_n=len(p3_inner_va),
        fixed_inner_split=True,full_dev_refit=False,
        best_epoch=best_epoch,best_val_auc_hist=float(h['val_auc'].max()),
        train_auc=train_auc,validation_auc=val_auc,
        fit_seconds=float(elapsed),**met))
    p3_preds.append(pd.DataFrame({
        'seed':seed,'original_index':p3_test_ids,
        'hostname':p3_hosts_test,'label':p3_y_test,
        'cnn_probability':proba,'cnn_prediction':(proba>=P3_THRESHOLD).astype(int),
        'raw_pixel_probability':p3_raw_probs,
        'lexical_probability':p3_lex_probs,
    }))
    model.save_weights(str(P3_OUT/f'cnn_seed_{seed}.weights.h5'))
    del model,history
    keras.backend.clear_session()
    gc.collect()

p3_cnn_metrics=pd.DataFrame(p3_seed_rows)
p3_cnn_metrics.to_csv(P3_OUT/'cnn_matched_metrics_by_seed.csv',index=False)
p3_predictions=pd.concat(p3_preds,ignore_index=True)
p3_predictions.to_csv(P3_OUT/'cnn_matched_predictions.csv',index=False)
print('\n=== TRAINING COMPLETE — ALL SEEDS ===')
print(p3_cnn_metrics[['seed','train_auc','validation_auc','auc','f1','mcc',
                      'accuracy','predicted_phishing_rate']]
      .round(4).to_string(index=False))

In [ ]:
p3_comparison=pd.DataFrame([
    dict(model='Trad-style raw-pixel XGBoost',seed='fixed',n=1777,
         **p3_metrics(p3_y_test,p3_raw_probs)),
    *[dict(model='Image-only Residual CNN (clean matched protocol)',
           **row) for row in p3_seed_rows],
    dict(model='10-feature lexical XGBoost',seed='fixed',n=1777,
         **p3_metrics(p3_y_test,p3_lex_probs)),
])
p3_comparison.to_csv(P3_OUT/'matched_3model_comparison.csv',index=False)
print('=== FINAL THREE MODEL COMPARISON ===')
print(p3_comparison[['model','seed','n','auc','f1','mcc','accuracy']]
      .round(4).to_string(index=False))
print('\n=== CNN ACROSS ALL SEEDS (INCLUDING FAILURES) ===')
for metric in ('auc','f1','mcc','accuracy'):
    print(f'{metric}: {p3_cnn_metrics[metric].mean():.4f} '
          f'+/- {p3_cnn_metrics[metric].std(ddof=1):.4f}')

In [ ]:
p3_uniq,p3_inverse=np.unique(p3_hosts_test,return_inverse=True)
p3_rows_by_host=[np.flatnonzero(p3_inverse==k) for k in range(len(p3_uniq))]
p3_ci_rows=[]
for seed in P3_SEEDS:
    for name,prob_a,prob_b in [
        ('Lexical_minus_CNN',p3_lex_probs,p3_scores[seed]),
        ('CNN_minus_RawXGB',p3_scores[seed],p3_raw_probs)]:
        observed=float(roc_auc_score(p3_y_test,prob_a)-
                       roc_auc_score(p3_y_test,prob_b))
        rng=np.random.default_rng(70000+seed)
        deltas=[]
        for _ in range(P3_BOOTSTRAPS):
            chosen=rng.integers(0,len(p3_uniq),size=len(p3_uniq))
            idx=np.concatenate([p3_rows_by_host[k] for k in chosen])
            yy=p3_y_test[idx]
            if len(np.unique(yy)) != 2:
                continue
            deltas.append(roc_auc_score(yy,prob_a[idx])-
                          roc_auc_score(yy,prob_b[idx]))
        assert len(deltas)>=P3_BOOTSTRAPS//2
        lo,hi=np.quantile(deltas,[0.025,0.975])
        p3_ci_rows.append(dict(seed=seed,contrast=name,
            auc_delta=observed,ci_low=float(lo),ci_high=float(hi),
            valid_replicates=len(deltas)))
p3_ci=pd.DataFrame(p3_ci_rows)
p3_ci.to_csv(P3_OUT/'paired_hostname_auc_ci_by_seed.csv',index=False)
print('=== PAIRED HOSTNAME-BOOTSTRAP AUC CONTRASTS ===')
print(p3_ci.round(4).to_string(index=False))

In [ ]:
p3_uniq,p3_inverse=np.unique(p3_hosts_test,return_inverse=True)
p3_rows_by_host=[np.flatnonzero(p3_inverse==k) for k in range(len(p3_uniq))]
p3_ci_rows=[]
for seed in P3_SEEDS:
    for name,prob_a,prob_b in [
        ('Lexical_minus_CNN',p3_lex_probs,p3_scores[seed]),
        ('CNN_minus_RawXGB',p3_scores[seed],p3_raw_probs)]:
        observed=float(roc_auc_score(p3_y_test,prob_a)-
                       roc_auc_score(p3_y_test,prob_b))
        rng=np.random.default_rng(70000+seed)
        deltas=[]
        for _ in range(P3_BOOTSTRAPS):
            chosen=rng.integers(0,len(p3_uniq),size=len(p3_uniq))
            idx=np.concatenate([p3_rows_by_host[k] for k in chosen])
            yy=p3_y_test[idx]
            if len(np.unique(yy)) != 2:
                continue
            deltas.append(roc_auc_score(yy,prob_a[idx])-
                          roc_auc_score(yy,prob_b[idx]))
        assert len(deltas)>=P3_BOOTSTRAPS//2
        lo,hi=np.quantile(deltas,[0.025,0.975])
        p3_ci_rows.append(dict(seed=seed,contrast=name,
            auc_delta=observed,ci_low=float(lo),ci_high=float(hi),
            valid_replicates=len(deltas)))
p3_ci=pd.DataFrame(p3_ci_rows)
p3_ci.to_csv(P3_OUT/'paired_hostname_auc_ci_by_seed.csv',index=False)
print('=== PAIRED HOSTNAME-BOOTSTRAP AUC CONTRASTS ===')
print(p3_ci.round(4).to_string(index=False))

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.6))
labels=['Raw XGB']+[f'CNN seed {s}' for s in P3_SEEDS]+['Lexical XGB']
auc_values=[p3_raw_auc]+[float(p3_cnn_metrics.loc[
    p3_cnn_metrics['seed']==s,'auc'].iloc[0]) for s in P3_SEEDS]+[p3_lex_auc]
ax.bar(labels,auc_values)
ax.set_ylim(0,1);ax.set_ylabel('ROC–AUC (1,777 matched holdout samples)')
ax.set_title('Matched representation and model comparison')
for i,v in enumerate(auc_values): ax.text(i,v+0.018,f'{v:.3f}',ha='center',fontsize=9)
ax.tick_params(axis='x',labelrotation=16)
fig.tight_layout()
fig.savefig(P3_OUT/'auc_model_comparison.png',dpi=160)
plt.show()

fig,ax=plt.subplots(figsize=(8,4.6))
for seed in P3_SEEDS:
    h=p3_histories[seed]
    ax.plot(np.arange(1,len(h)+1),h['val_auc'],label=f'Seed {seed}')
ax.set_xlabel('Epoch');ax.set_ylabel('Validation AUC (Keras)')
ax.set_ylim(0,1);ax.set_title('CNN training stability on the same inner validation set')
ax.legend();fig.tight_layout()
fig.savefig(P3_OUT/'validation_auc_by_seed.png',dpi=160)
plt.show()

p3_protocol={
    'experiment':'Part 3 Clean V2: matched image-only Residual CNN',
    'model_status':'Method-based residual CNN; not exact Do et al. architecture reproduction',
    'source_baseline_notebook':'Baseline & Literature Reproduction.ipynb, cells 18, 20-24',
    'original_aligned_samples':len(p3_df),
    'outer_split':'GroupShuffleSplit(test_size=0.20, random_state=42, group=hostname)',
    'development_n':len(p3_dev_ix),'matched_holdout_n':len(p3_test_ix),
    'exact_exp6_row_alignment':True,
    'fixed_inner_split':'200 grouped candidate splits, fixed seed 20261108, optimized class prevalence difference + size; DEV ONLY',
    'fixed_inner_train_n':len(p3_inner_tr),'fixed_inner_validation_n':len(p3_inner_va),
    'inner_train_positive_rate':float(p3_y_dev[p3_inner_tr].mean()),
    'inner_val_positive_rate':float(p3_y_dev[p3_inner_va].mean()),
    'seeds':list(P3_SEEDS),'full_development_refit':False,
    'preprocessing':'per-QR gaussian sigma=0.5, per-image min-max, 69x69x1',
    'architecture':'3 32-filter residual blocks, dense 128, dropout 0.3',
    'augmentation':'random rotation +/-10 degrees; GaussianNoiseClip stddev 0.1 (training only)',
    'training_stability_changes_vs_old':'AdamW constant learning rate 3e-4, clipnorm 1.0, ReduceLROnPlateau, patience 12, fixed group-aware validation, no refit',
    'max_epochs':P3_MAX_EPOCHS,'batch_size':P3_BATCH_SIZE,
    'classification_threshold':P3_THRESHOLD,
    'test_label_use':'Only final evaluation (never gradient updates, split search, threshold selection or early stopping)',
    'holdout_caveat':'This holdout was previously inspected during CNN debugging; treat CNN comparison as exploratory and disclose the reuse.',
    'input_caveat':'CNN and raw XGB use QR pixels; lexical XGB uses decoded URL. Matched test rows do not mean identical information availability.',
    'training_sample_caveat':'CNN trained on fixed inner training subset, not all 7,936 development samples.',
    'tensorflow_version':tf.__version__,'python_version':platform.python_version(),
}
with open(P3_OUT/'experiment_protocol.json','w',encoding='utf-8') as fh:
    json.dump(p3_protocol,fh,indent=2,ensure_ascii=False)
print('\nALL DONE. Saved to:',P3_OUT.resolve())
for file in sorted(P3_OUT.iterdir()): print('  ',file.name)
print('\nINTERPRETATION RULE: report ALL seeds (including collapsed runs). Do not describe'
      '\nthis as an exact Do et al. reproduction or as proving lexical models universally beat CNNs.')

In [ ]:

import pandas as pd
from pathlib import Path

folder = Path("part3_cnn_matched_clean_results")

df = pd.read_csv(
    folder / "cnn_matched_metrics_by_seed.csv"
)

cols = [
    "seed",
    "best_epoch",
    "train_auc",
    "validation_auc",
    "auc",
    "probability_std",
    "predicted_phishing_rate"
]

print("=== CNN TRAINING STABILITY AUDIT ===")
print(df[cols].round(4).to_string(index=False))


A representative image-only residual CNN was evaluated on the same 1,777-sample hostname-disjoint holdout as the lexical and raw-pixel XGBoost baselines. Across three random seeds, CNN ROC–AUC varied substantially (0.8531, 0.5000, and 0.7675), with one training run failing to learn a discriminative representation. The lexical XGBoost achieved a higher ROC–AUC of 0.9509 under the evaluated conditions. These findings are treated as an exploratory comparison rather than evidence of universal superiority over CNN-based methods.